# Install dependencies (approximately 5 mins + might need to restart)

In [ ]:
!pip install nemo_toolkit['all'] wget
!apt-get install -y ffmpeg

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


# Verify GPU

In [ ]:
import torch

if torch.cuda.is_available():
    print(f"✅ GPU ready: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("❌ No GPU — go to Runtime > Change runtime type > T4 GPU")

/usr/local/lib/python3.12/dist-packages/torch/cuda/__init__.py:65: FutureWarning: The pynvml package is deprecated. Please install nvidia-ml-py instead. If you did not install pynvml directly, please report this to the maintainers of the package that installed pynvml for you.
  import pynvml  # type: ignore[import]


❌ No GPU — go to Runtime > Change runtime type > T4 GPU


# Mount GG Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# All output will be saved here
OUTPUT_DIR = "/content/drive/MyDrive/diarization_output"

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Output will be saved to: {OUTPUT_DIR}")

Mounted at /content/drive
✅ Output will be saved to: /content/drive/MyDrive/diarization_output


# Upload audio file

In [ ]:
from google.colab import files
import shutil

print("Upload your audio file (wav, mp3, mp4, m4a)...")
uploaded = files.upload()

# Get the uploaded filename
original_filename = list(uploaded.keys())[0]
audio_input_path = f"/content/{original_filename}"
print(f"✅ Uploaded: {audio_input_path}")

Upload your audio file (wav, mp3, mp4, m4a)...


Saving sound1.m4a to sound1.m4a
✅ Uploaded: /content/sound1.m4a


# Convert Audio to 16kHz Mono WAV

In [ ]:
audio_wav_path = "/content/audio_input.wav"

!ffmpeg -y -i "{audio_input_path}" -ar 16000 -ac 1 "{audio_wav_path}"

# Quick check
import subprocess
result = subprocess.run(
    ["ffprobe", "-v", "error", "-show_entries",
     "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", audio_wav_path],
    capture_output=True, text=True
)
duration = float(result.stdout.strip())
print(f"✅ Audio ready — Duration: {duration:.1f}s ({duration/60:.1f} min)")

ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

# Create Manifest File

**Manifest File** is a simple text file (usually in .json or .jsonl format) that acts as an index or "map" for your dataset.

Instead of loading heavy audio files directly into the AI, you provide this manifest file so the system knows exactly where the data is and how to handle it.

**Efficiency**: It allows the AI to scan thousands of files instantly without opening the actual audio until they are needed for processing.

**Why is it used?**

It is the standard input format for NeMo models. Whether you are doing Speech-to-Text (ASR) or Speaker Diarization, the manifest tells the model which files to read, where they start, and what labels are associated with them.

In [ ]:
import json

manifest_path = "/content/diar_manifest.json"

manifest_entry = {
    "audio_filepath": audio_wav_path,
    "offset": 0, # begin at 0 second
    "duration": None,
    "label": "infer", # inference
    "text": "-",
    "num_speakers": None   # ← change to e.g. 2 if you know the speaker count
}

with open(manifest_path, "w") as f:
    json.dump(manifest_entry, f)
    f.write("\n")

print(f"✅ Manifest created: {manifest_path}")

✅ Manifest created: /content/diar_manifest.json


# Download Config & Run Diarization

In [ ]:
import wget
from omegaconf import OmegaConf
from nemo.collections.asr.models import ClusteringDiarizer

# Download NeMo meeting config
config_path = "/content/diar_infer_meeting.yaml"
if not os.path.exists(config_path):
    wget.download(
        "https://raw.githubusercontent.com/NVIDIA/NeMo/main/examples/"
        "speaker_tasks/diarization/conf/inference/diar_infer_meeting.yaml",
        config_path
    )

cfg = OmegaConf.load(config_path)

# ── Paths ───────────────────────────────────────────
cfg.diarizer.manifest_filepath = manifest_path
cfg.diarizer.out_dir           = OUTPUT_DIR

# ── VAD (MarbleNet) ─────────────────────────────────
cfg.diarizer.vad.model_path                      = "vad_multilingual_marblenet"
cfg.diarizer.vad.parameters.onset                = 0.8
cfg.diarizer.vad.parameters.offset               = 0.6
cfg.diarizer.vad.parameters.min_duration_on      = 0.1
cfg.diarizer.vad.parameters.min_duration_off     = 0.4

# ── TitaNet-L speaker embeddings ────────────────────
cfg.diarizer.speaker_embeddings.model_path                          = "titanet_large" # TitaNet-L
cfg.diarizer.speaker_embeddings.parameters.window_length_in_sec     = [1.5, 1.25, 1.0, 0.75, 0.5]
cfg.diarizer.speaker_embeddings.parameters.shift_length_in_sec      = [0.75, 0.625, 0.5, 0.375, 0.25]
cfg.diarizer.speaker_embeddings.parameters.multiscale_weights        = [1, 1, 1, 1, 1]

# ── Clustering ──────────────────────────────────────
cfg.diarizer.clustering.parameters.oracle_num_speakers = False
cfg.diarizer.clustering.parameters.max_num_speakers    = 8 # max_num_speakers

# ── Run ─────────────────────────────────────────────
print("🚀 Starting diarization...")
diarizer = ClusteringDiarizer(cfg=cfg)
diarizer.diarize()
print("✅ Diarization complete!")

🚀 Starting diarization...
[NeMo I 2026-04-10 15:42:31 clustering_diarizer:117] Loading pretrained vad_multilingual_marblenet model from NGC
[NeMo I 2026-04-10 15:42:31 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/vad_multilingual_marblenet/versions/1.10.0/files/vad_multilingual_marblenet.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo
[NeMo I 2026-04-10 15:42:32 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-04-10 15:42:32 classification_models:641] Please use the EncDecSpeakerLabelModel instead of this model. EncDecClassificationModel model is kept for backward compatibility with older models.
[NeMo W 2026-04-10 15:42:32 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/ami_train_0.63.json,/manifests/freesound_background_train.json,/manifests/freesound_laughter_train.json,/manifests/fisher_2004_background.json,/manifests/fisher_2004_speech_sampled.json,/manifests/google_train_manifest.json,/manifests/icsi_all_0.63.json,/manifests/musan_freesound_train.json,/manifests/musan_music_train.json,/manifests/musan_soundbible_train.json,/manifests/mandarin_train_sample.json,/manifests/german_train_sample.json,/manifests/spanish_train_sample.json,/manifests/french_train_sample.json,/manifests/russian_tr

[NeMo I 2026-04-10 15:42:33 save_restore_connector:285] Model EncDecClassificationModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/vad_multilingual_marblenet/670f425c7f186060b7a7268ba6dfacb2/vad_multilingual_marblenet.nemo.
[NeMo I 2026-04-10 15:42:33 clustering_diarizer:150] Loading pretrained titanet_large model from NGC
[NeMo I 2026-04-10 15:42:33 cloud:68] Downloading from: https://api.ngc.nvidia.com/v2/models/nvidia/nemo/titanet_large/versions/v1/files/titanet-l.nemo to /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo
[NeMo I 2026-04-10 15:42:35 common:939] Instantiating model from pre-trained checkpoint


[NeMo W 2026-04-10 15:42:36 modelPT:188] If you intend to do training or fine-tuning, please call the ModelPT.setup_training_data() method and provide a valid configuration file to setup the train data loader.
    Train config : 
    manifest_filepath: /manifests/combined_fisher_swbd_voxceleb12_librispeech/train.json
    sample_rate: 16000
    labels: null
    batch_size: 64
    shuffle: true
    is_tarred: false
    tarred_audio_filepaths: null
    tarred_shard_strategy: scatter
    augmentor:
      noise:
        manifest_path: /manifests/noise/rir_noise_manifest.json
        prob: 0.5
        min_snr_db: 0
        max_snr_db: 15
      speed:
        prob: 0.5
        sr: 16000
        resample_type: kaiser_fast
        min_speed_rate: 0.95
        max_speed_rate: 1.05
    num_workers: 15
    pin_memory: true
    
[NeMo W 2026-04-10 15:42:36 modelPT:195] If you intend to do validation, please call the ModelPT.setup_validation_data() or ModelPT.setup_multiple_validation_data() method 

[NeMo I 2026-04-10 15:42:37 save_restore_connector:285] Model EncDecSpeakerLabelModel was successfully restored from /root/.cache/torch/NeMo/NeMo_2.7.2/titanet-l/11ba0924fdf87c049e339adbf6899d48/titanet-l.nemo.


...

[NeMo I 2026-04-10 16:40:08 clustering_diarizer:383] Saved embedding files to /content/drive/MyDrive/diarization_output/speaker_outputs/embeddings



[NeMo W 2026-04-10 16:40:08 speaker_utils:473] cuda=False, using CPU for eigen decomposition. This might slow down the clustering process.
clustering: 100%|██████████| 1/1 [00:06<00:00,  6.22s/it]

[NeMo I 2026-04-10 16:40:14 clustering_diarizer:451] Outputs are saved in /content/drive/MyDrive/diarization_output directory



[NeMo W 2026-04-10 16:40:14 der:221] Check if each ground truth RTTMs were present in the provided manifest file. Skipping calculation of Diariazation Error Rate


✅ Diarization complete!


# Read & Print Results

In [ ]:
import glob

# Find the RTTM file
rttm_files = glob.glob(os.path.join(OUTPUT_DIR, "pred_rttms", "*.rttm"))

if not rttm_files:
    print("❌ No RTTM file found. Check the output directory.")
else:
    rttm_path = rttm_files[0]
    print(f"📄 Reading: {rttm_path}\n")
    print(f"{'Start':>8}  {'End':>8}  {'Duration':>10}  Speaker")
    print("-" * 45)

    segments = []
    with open(rttm_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            start    = float(parts[3])
            duration = float(parts[4])
            speaker  = parts[7]
            end      = start + duration
            segments.append((start, end, duration, speaker))
            print(f"{start:>7.2f}s  {end:>7.2f}s  {duration:>9.2f}s  {speaker}")

    print(f"\n✅ Total segments: {len(segments)}")
    print(f"   Unique speakers: {len(set(s[3] for s in segments))}")

📄 Reading: /content/drive/MyDrive/diarization_output/pred_rttms/audio_input.rttm

   Start       End    Duration  Speaker
---------------------------------------------
   0.00s    12.88s      12.88s  speaker_2
  12.88s    37.38s      24.50s  speaker_0
  37.38s    38.62s       1.25s  speaker_2
  38.62s    39.71s       1.08s  speaker_0
  40.16s    41.00s       0.84s  speaker_0
  41.64s    65.65s      24.01s  speaker_0
  66.21s    85.27s      19.06s  speaker_0
  85.91s   130.72s      44.81s  speaker_0
 131.26s   134.67s       3.41s  speaker_0
 135.13s   146.49s      11.36s  speaker_0
 147.33s   149.88s       2.55s  speaker_0
 150.51s   150.76s       0.25s  speaker_0
 152.28s   155.41s       3.12s  speaker_3
 155.41s   158.16s       2.75s  speaker_1
 158.16s   161.66s       3.50s  speaker_3
 161.66s   167.66s       6.00s  speaker_1
 167.66s   168.91s       1.25s  speaker_3
 168.91s   178.95s      10.04s  speaker_1
 179.70s   180.82s       1.12s  speaker_3
 180.82s   191.35s      10.53s  sp